# Fixed-geometry off-design map and TESPy export

**KCORC Summer School 2026 — instructor development draft (v0.1)**

This notebook starts from the frozen design exported by `01_supersonic_axial_orc_stage_design.ipynb` and generates a reduced-order operating map.

## Learning objectives

Students should be able to:

1. distinguish **design** from **off-design evaluation**;
2. keep the turbine geometry fixed while varying operating conditions;
3. calculate choked stator mass-flow capacity;
4. evaluate the effect of pressure ratio and rotor speed on velocity triangles and efficiency;
5. create and interpolate a two-dimensional performance map;
6. normalize the map for transfer to a cycle model;
7. export a transparent CSV contract for the following TESPy workshop.

> The map is intentionally a reduced-order teaching model. It is not a validated replacement for CFD or experimental performance maps.


## 0. Key modeling decision

The first notebook sizes the machine once. Here the following quantities remain frozen:

- throat and outlet areas;
- mean diameter and blade height;
- stator and rotor angles;
- chord, pitch, blade count, and partial admission.

The operating point changes, but the metal does not.

In this preliminary map:

- outlet pressure is fixed;
- inlet pressure is varied to change pressure ratio;
- the inlet state keeps approximately the **design superheat above the dew line**, so the working fluid remains in the vapor region over the selected subcritical range;
- the stator throat determines the choked mass-flow capacity;
- rotational speed changes the rotor velocity triangle, power, and efficiency;
- pressure ratio is imposed by the cycle boundary conditions rather than directly by rotor speed.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import LinearNDInterpolator

try:
    import CoolProp.CoolProp as CP
except ImportError as exc:
    raise ImportError(
        "CoolProp is required. In the repository terminal run: uv add CoolProp"
    ) from exc

plt.rcParams.update({
    "figure.figsize": (8.5, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
DESIGN_FILE = OUTPUT_DIR / "design_point.json"

if not DESIGN_FILE.exists():
    raise FileNotFoundError(
        "Run 01_supersonic_axial_orc_stage_design.ipynb first. "
        "It must create outputs/design_point.json."
    )

with DESIGN_FILE.open("r", encoding="utf-8") as file:
    design = json.load(file)

design


## 1. Recover the frozen design

The JSON file is the interface between the design and off-design notebooks. Keeping this interface explicit will also make later TESPy coupling easier.


In [ ]:
fluid = design["inputs"]["fluid"]
T_in_design_K = design["inputs"]["T_in_K"]
p_out_design_Pa = design["inputs"]["p_out_Pa"]
p_in_design_Pa = design["inputs"]["p_in_Pa"]

PR_design = design["thermodynamics"]["pressure_ratio_design"]
n_design_rpm = design["performance"]["n_design_rpm"]
m_dot_design_kg_s = design["performance"]["m_dot_design_kg_s"]
eta_design = design["performance"]["eta_turb_design"]
P_design_W = design["performance"]["P_mech_design_W"]

A_throat_m2 = design["nozzle"]["A_throat_m2"]
A_outlet_m2 = design["nozzle"]["A_outlet_act_m2"]

D_mid_m = design["geometry"]["D_mid_m"]
h_rotor_m = design["geometry"]["h_rotor_m"]
alpha_stator_deg = design["geometry"]["alpha_stator_deg"]
beta_rotor_in_deg = design["geometry"]["beta_rotor_in_deg"]
beta_rotor_out_acute_deg = design["geometry"]["beta_rotor_out_acute_deg"]
chord_rotor_m = design["geometry"]["chord_rotor_m"]
pitch_rotor_m = design["geometry"]["pitch_rotor_m"]
partial_admission = design["geometry"]["partial_admission"]
K_sector = design["geometry"]["K_sector"]

p_critical_Pa = float(CP.PropsSI("pcrit", fluid))
T_dew_design_K = float(CP.PropsSI("T", "P", p_in_design_Pa, "Q", 1.0, fluid))
design_superheat_K = T_in_design_K - T_dew_design_K

pd.DataFrame([
    ("Design pressure ratio", PR_design, "-"),
    ("Design rotational speed", n_design_rpm, "rpm"),
    ("Design mass flow", m_dot_design_kg_s, "kg/s"),
    ("Design superheat", design_superheat_K, "K"),
    ("Fluid critical pressure", p_critical_Pa / 1e6, "MPa"),
    ("Frozen throat area", A_throat_m2 * 1e6, "mm^2"),
    ("Frozen outlet area", A_outlet_m2 * 1e6, "mm^2"),
    ("Frozen mean diameter", D_mid_m * 1e3, "mm"),
    ("Frozen rotor height", h_rotor_m * 1e3, "mm"),
], columns=["Quantity", "Value", "Unit"])


## 2. Property and loss helper functions

These reproduce the same reduced-order equations used in the design notebook. Keeping the equations consistent is more important here than adding additional complexity.


In [ ]:
def state_PT(p_Pa: float, T_K: float, fluid: str) -> dict[str, float]:
    return {
        "h_J_kg": float(CP.PropsSI("H", "P", p_Pa, "T", T_K, fluid)),
        "s_J_kgK": float(CP.PropsSI("S", "P", p_Pa, "T", T_K, fluid)),
        "rho_kg_m3": float(CP.PropsSI("D", "P", p_Pa, "T", T_K, fluid)),
        "a_m_s": float(CP.PropsSI("A", "P", p_Pa, "T", T_K, fluid)),
    }


def state_PS(p_Pa: float, s_J_kgK: float, fluid: str) -> dict[str, float]:
    return {
        "h_J_kg": float(CP.PropsSI("H", "P", p_Pa, "S", s_J_kgK, fluid)),
        "T_K": float(CP.PropsSI("T", "P", p_Pa, "S", s_J_kgK, fluid)),
        "rho_kg_m3": float(CP.PropsSI("D", "P", p_Pa, "S", s_J_kgK, fluid)),
        "a_m_s": float(CP.PropsSI("A", "P", p_Pa, "S", s_J_kgK, fluid)),
    }


def state_PH(p_Pa: float, h_J_kg: float, fluid: str) -> dict[str, float]:
    return {
        "h_J_kg": float(h_J_kg),
        "T_K": float(CP.PropsSI("T", "P", p_Pa, "H", h_J_kg, fluid)),
        "s_J_kgK": float(CP.PropsSI("S", "P", p_Pa, "H", h_J_kg, fluid)),
        "rho_kg_m3": float(CP.PropsSI("D", "P", p_Pa, "H", h_J_kg, fluid)),
        "a_m_s": float(CP.PropsSI("A", "P", p_Pa, "H", h_J_kg, fluid)),
    }


def stator_velocity_coefficient(Ma_out_is: float) -> float:
    loss_fraction = (
        0.0029 * Ma_out_is**3
        - 0.0502 * Ma_out_is**2
        + 0.2241 * Ma_out_is
        - 0.0877
    )
    return float(np.sqrt(np.clip(1.0 - loss_fraction, 1e-6, 1.0)))


def rotor_velocity_coefficient(
    theta_deg: float,
    Ma1_rel: float,
    blade_height_m: float,
    chord_m: float,
) -> float:
    phi_base = (
        0.957
        - 0.000362 * theta_deg
        - 0.0258 * Ma1_rel
        + 0.00000639 * theta_deg**2
        + 0.0674 * Ma1_rel**2
        - 0.0000000753 * theta_deg**3
        - 0.043 * Ma1_rel**3
        - 0.000238 * theta_deg * Ma1_rel
        + 0.00000145 * theta_deg**2 * Ma1_rel
        + 0.0000425 * theta_deg * Ma1_rel**2
    )
    k1 = 2.0
    k2 = 0.65
    inside = 1.0 - ((1.0 - phi_base**2) / k1) * (
        1.0 + (k1 - 1.0) * (blade_height_m / chord_m) ** (-k2)
    )
    return float(np.sqrt(max(inside, 1e-12)))


## 3. Stator capacity at an off-design pressure ratio

For each inlet pressure, an isentropic pressure path is constructed. If it crosses \(Ma=1\), the stator is treated as choked and the fixed throat area gives

\[
\dot m=A^*\rho^*a^*.
\]

This captures the main mass-flow governance of a fixed convergent-divergent stator.

The imposed outlet pressure and frozen outlet area will generally not be perfectly compatible away from the design point. The notebook therefore reports an **outlet-area continuity mismatch** as a diagnostic. A large mismatch indicates under-expansion, over-expansion, shocks, or other physics not resolved by this reduced model.


In [ ]:
def evaluate_stator_offdesign(
    p_in_Pa: float,
    T_in_K: float,
    p_out_Pa: float,
    fluid: str,
    A_throat_m2: float,
    A_outlet_m2: float,
    n_steps: int = 180,
) -> dict[str, float | bool]:
    inlet = state_PT(p_in_Pa, T_in_K, fluid)
    p_path = np.linspace(p_in_Pa, p_out_Pa, n_steps)

    h_is = np.empty_like(p_path)
    rho_is = np.empty_like(p_path)
    a_is = np.empty_like(p_path)

    for i, p_i in enumerate(p_path):
        st = state_PS(float(p_i), inlet["s_J_kgK"], fluid)
        h_is[i] = st["h_J_kg"]
        rho_is[i] = st["rho_kg_m3"]
        a_is[i] = st["a_m_s"]

    c_is = np.sqrt(np.maximum(2.0 * (inlet["h_J_kg"] - h_is), 0.0))
    Ma_is = np.divide(c_is, a_is, out=np.zeros_like(c_is), where=a_is > 0.0)
    choked = bool(np.nanmax(Ma_is) >= 1.0)

    if not choked:
        return {
            "choked": False,
            "valid_stator": False,
            "m_dot_kg_s": np.nan,
            "dh_is_J_kg": float(inlet["h_J_kg"] - h_is[-1]),
            "Ma_out_is": float(Ma_is[-1]),
            "Ma_out": np.nan,
            "c1_m_s": np.nan,
            "rho1_kg_m3": np.nan,
            "a1_m_s": np.nan,
            "h1_J_kg": np.nan,
            "area_mismatch_rel": np.nan,
            "phi_stator": np.nan,
            "h_in_J_kg": inlet["h_J_kg"],
            "s_in_J_kgK": inlet["s_J_kgK"],
        }

    throat_index = int(np.nanargmin(np.abs(Ma_is - 1.0)))
    m_dot = A_throat_m2 * rho_is[throat_index] * c_is[throat_index]

    phi_s = stator_velocity_coefficient(float(Ma_is[-1]))
    h1 = h_is[throat_index] - phi_s**2 * (h_is[throat_index] - h_is[-1])
    state_1 = state_PH(p_out_Pa, float(h1), fluid)
    c1 = float(np.sqrt(max(2.0 * (inlet["h_J_kg"] - h1), 0.0)))
    Ma1 = c1 / state_1["a_m_s"]

    m_dot_supported_by_outlet = A_outlet_m2 * state_1["rho_kg_m3"] * c1
    area_mismatch_rel = (m_dot_supported_by_outlet - m_dot) / m_dot

    return {
        "choked": True,
        "valid_stator": True,
        "m_dot_kg_s": float(m_dot),
        "dh_is_J_kg": float(inlet["h_J_kg"] - h_is[-1]),
        "Ma_out_is": float(Ma_is[-1]),
        "Ma_out": float(Ma1),
        "c1_m_s": c1,
        "rho1_kg_m3": state_1["rho_kg_m3"],
        "a1_m_s": state_1["a_m_s"],
        "h1_J_kg": float(h1),
        "area_mismatch_rel": float(area_mismatch_rel),
        "phi_stator": phi_s,
        "h_in_J_kg": inlet["h_J_kg"],
        "s_in_J_kgK": inlet["s_J_kgK"],
    }


## 4. Rotor evaluation at an off-design speed

For a given stator operating point, rotor speed changes \(U\), the relative inlet triangle, rotor loss coefficient, outlet swirl, Euler work, and loss powers.

The model does **not** force the turbine onto its optimum-speed line. The optimum is extracted later from the calculated map.


In [ ]:
def evaluate_rotor_offdesign(
    stator: dict[str, float | bool],
    n_rpm: float,
    p_out_Pa: float,
) -> dict[str, float | bool]:
    if not bool(stator["valid_stator"]):
        return {"valid": False, "eta_turb": np.nan, "P_mech_W": np.nan}

    m_dot = float(stator["m_dot_kg_s"])
    c1 = float(stator["c1_m_s"])
    dh_is = float(stator["dh_is_J_kg"])

    alpha_1_rad = np.radians(alpha_stator_deg)
    c1a = c1 * np.sin(alpha_1_rad)
    c1u = c1 * np.cos(alpha_1_rad)

    U = np.pi * D_mid_m * n_rpm / 60.0
    w1a = c1a
    w1u = c1u - U
    w1 = float(np.hypot(w1a, w1u))
    Ma1_rel = w1 / float(stator["a1_m_s"])

    beta2_deg = 180.0 - beta_rotor_out_acute_deg
    theta_deg = beta2_deg - beta_rotor_in_deg
    phi_rotor = rotor_velocity_coefficient(
        theta_deg,
        Ma1_rel,
        h_rotor_m,
        chord_rotor_m,
    )

    w2_signed = -w1 * phi_rotor * K_sector
    w2a = w2_signed * (-np.sin(np.radians(beta2_deg)))
    w2u = w2_signed * (-np.cos(np.radians(beta2_deg)))
    c2a = w2a
    c2u = w2u + U
    c2 = float(np.hypot(c2a, c2u))

    dh_rotor_loss = (w1**2 - w2_signed**2) / 2.0
    h2 = float(stator["h1_J_kg"]) + dh_rotor_loss

    try:
        state_2 = state_PH(p_out_Pa, h2, fluid)
    except ValueError:
        return {"valid": False, "eta_turb": np.nan, "P_mech_W": np.nan}

    rho2 = state_2["rho_kg_m3"]
    P_aero = m_dot * U * (c1u - c2u)
    P_fric = 0.01 * (n_rpm / 60.0) ** 3 * D_mid_m**5 * rho2

    P_pa_legacy = (
        (1.0 - partial_admission)
        * rho2
        * (n_rpm / 60.0) ** 3
        * D_mid_m**4
        * 3.8
        * h_rotor_m
    )
    Kp = 3.63 * 0.4
    P_pa_pumping = (
        Kp
        * rho2
        * U**3
        * h_rotor_m
        * D_mid_m**4
        * (1.0 - partial_admission)
    )
    P_pa = max(P_pa_legacy, P_pa_pumping)

    P_mech = P_aero - P_fric - P_pa
    eta_turb = P_mech / (m_dot * dh_is)
    U_over_c_is = U / np.sqrt(2.0 * dh_is)

    finite = np.isfinite([P_mech, eta_turb, phi_rotor, c2a, c2u]).all()
    plausible = (
        finite
        and P_mech > 0.0
        and 0.0 < eta_turb < 0.90
        and phi_rotor > 0.0
        and c2a > 0.0
    )

    return {
        "valid": bool(plausible),
        "n_rpm": float(n_rpm),
        "U_m_s": float(U),
        "U_over_c_is": float(U_over_c_is),
        "Ma1_rel": float(Ma1_rel),
        "phi_rotor": float(phi_rotor),
        "c1a_m_s": float(c1a),
        "c1u_m_s": float(c1u),
        "c2a_m_s": float(c2a),
        "c2u_m_s": float(c2u),
        "c2_m_s": float(c2),
        "P_aero_W": float(P_aero),
        "P_fric_W": float(P_fric),
        "P_partial_admission_W": float(P_pa),
        "P_mech_W": float(P_mech),
        "eta_turb": float(eta_turb),
        "T_out_K": float(state_2["T_K"]),
    }


## 5. Build the map

The default range is local to the subcritical design:

\[
0.5\lesssim \Pi/\Pi_d\lesssim 1.35,
\qquad
0.5\lesssim n/n_d\lesssim 2.0.
\]

The upper pressure-ratio limit is also capped below the working-fluid critical pressure. At each inlet pressure, the inlet temperature is set to the local dew temperature plus the design-point superheat.

The resolution is intentionally modest so the map can be calculated during a workshop. It can be increased for preparation of the final repository.


In [ ]:
PR_min = max(1.5, 0.5 * PR_design)
PR_max_subcritical = 0.95 * p_critical_Pa / p_out_design_Pa
PR_max = min(1.35 * PR_design, PR_max_subcritical)

if PR_max <= PR_min:
    raise ValueError("The selected pressure-ratio range is not valid for this fluid/design.")

PR_values = np.linspace(PR_min, PR_max, 34)
n_values_rpm = np.linspace(0.5 * n_design_rpm, 2.0 * n_design_rpm, 42)

rows: list[dict[str, float | bool]] = []

for PR in PR_values:
    p_in_Pa = PR * p_out_design_Pa

    try:
        T_dew_K = float(CP.PropsSI("T", "P", p_in_Pa, "Q", 1.0, fluid))
        T_in_K = T_dew_K + design_superheat_K
        stator = evaluate_stator_offdesign(
            p_in_Pa=p_in_Pa,
            T_in_K=T_in_K,
            p_out_Pa=p_out_design_Pa,
            fluid=fluid,
            A_throat_m2=A_throat_m2,
            A_outlet_m2=A_outlet_m2,
        )
    except ValueError:
        T_in_K = np.nan
        stator = {
            "valid_stator": False,
            "choked": False,
            "m_dot_kg_s": np.nan,
            "Ma_out_is": np.nan,
            "Ma_out": np.nan,
            "area_mismatch_rel": np.nan,
        }

    for n_rpm in n_values_rpm:
        rotor = evaluate_rotor_offdesign(stator, float(n_rpm), p_out_design_Pa)
        rows.append({
            "pressure_ratio": float(PR),
            "p_in_Pa": float(p_in_Pa),
            "T_in_K": float(T_in_K),
            "superheat_K": float(design_superheat_K),
            "p_out_Pa": float(p_out_design_Pa),
            "n_rpm": float(n_rpm),
            "m_dot_kg_s": float(stator.get("m_dot_kg_s", np.nan)),
            "choked": bool(stator.get("choked", False)),
            "Ma_nozzle_out_is": float(stator.get("Ma_out_is", np.nan)),
            "Ma_nozzle_out": float(stator.get("Ma_out", np.nan)),
            "area_mismatch_rel": float(stator.get("area_mismatch_rel", np.nan)),
            **rotor,
        })

map_df = pd.DataFrame(rows)
map_df.head()


## 6. Normalize the map

The following dimensionless coordinates are convenient for handoff to the cycle workshop:

\[
\Pi^*=\frac{\Pi}{\Pi_d},
\qquad
n^*=\frac{n}{n_d},
\qquad
\dot m^*=\frac{\dot m}{\dot m_d},
\qquad
P^*=\frac{P}{P_d}.
\]

This normalization makes interpolation and comparison easier, but it does **not** automatically make the map universal across different fluids, inlet temperatures, or turbine geometries.


In [ ]:
map_df["PR_rel"] = map_df["pressure_ratio"] / PR_design
map_df["n_rel"] = map_df["n_rpm"] / n_design_rpm
map_df["m_dot_rel"] = map_df["m_dot_kg_s"] / m_dot_design_kg_s
map_df["P_mech_rel"] = map_df["P_mech_W"] / P_design_W
map_df["eta_rel"] = map_df["eta_turb"] / eta_design

map_df[[
    "PR_rel",
    "n_rel",
    "m_dot_rel",
    "eta_turb",
    "P_mech_rel",
    "choked",
    "valid",
]].head()


## 7. Efficiency map

The black contours show turbine efficiency. The design point is marked separately. Invalid or non-physical reduced-model points are masked.


In [ ]:
eta_grid = map_df.pivot(index="pressure_ratio", columns="n_rpm", values="eta_turb")
valid_grid = map_df.pivot(index="pressure_ratio", columns="n_rpm", values="valid")

PR_grid = eta_grid.index.to_numpy(dtype=float)
n_grid = eta_grid.columns.to_numpy(dtype=float)
ETA = eta_grid.to_numpy(dtype=float)
VALID = valid_grid.to_numpy(dtype=bool)
ETA_masked = np.where(VALID, ETA, np.nan)

fig, ax = plt.subplots(figsize=(9.5, 6.5))
filled = ax.contourf(n_grid, PR_grid, ETA_masked, levels=20)

finite_eta = ETA_masked[np.isfinite(ETA_masked)]
if finite_eta.size:
    low = np.ceil(np.nanmin(finite_eta) / 0.025) * 0.025
    high = np.floor(np.nanmax(finite_eta) / 0.025) * 0.025
    levels = np.arange(low, high + 0.0125, 0.025)
    if levels.size >= 2:
        lines = ax.contour(n_grid, PR_grid, ETA_masked, levels=levels, linewidths=1.0)
        ax.clabel(lines, inline=True, fontsize=9, fmt="%.3f")

ax.scatter(
    [n_design_rpm],
    [PR_design],
    marker="x",
    s=100,
    label="Design point",
)
ax.set_xlabel(r"Rotational speed $n$ [rpm]")
ax.set_ylabel(r"Pressure ratio $\Pi$ [-]")
ax.set_title("Reduced-order fixed-geometry turbine efficiency map")
fig.colorbar(filled, ax=ax, label=r"Turbine efficiency $\eta_{is,t-s}$ [-]")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "offdesign_efficiency_map.png", dpi=250)
plt.show()


## 8. Mass-flow capacity and outlet-area mismatch

In this model, mass flow is determined by the fixed sonic throat and inlet state. It is therefore primarily a function of pressure ratio (through inlet pressure here), not rotor speed.

The outlet-area mismatch is a useful warning metric. Values far from zero indicate that the imposed back pressure and the design area ratio are no longer mutually compatible within a shock-free quasi-1D interpretation.


In [ ]:
capacity = (
    map_df.sort_values(["pressure_ratio", "n_rpm"])
    .drop_duplicates("pressure_ratio")
    [["pressure_ratio", "m_dot_kg_s", "m_dot_rel", "area_mismatch_rel", "choked"]]
)

fig, ax = plt.subplots()
ax.plot(capacity["pressure_ratio"], capacity["m_dot_kg_s"], marker="o", markersize=3)
ax.axvline(PR_design, linestyle="--", linewidth=1.0)
ax.axhline(m_dot_design_kg_s, linestyle="--", linewidth=1.0)
ax.set_xlabel(r"Pressure ratio $\Pi$ [-]")
ax.set_ylabel(r"Predicted choked mass flow $\dot m$ [kg/s]")
ax.set_title("Fixed-throat swallowing capacity")
plt.show()

fig, ax = plt.subplots()
ax.plot(capacity["pressure_ratio"], 100.0 * capacity["area_mismatch_rel"])
ax.axhline(0.0, linewidth=1.0)
ax.axvline(PR_design, linestyle="--", linewidth=1.0)
ax.set_xlabel(r"Pressure ratio $\Pi$ [-]")
ax.set_ylabel("Outlet continuity mismatch [%]")
ax.set_title("Diagnostic of imposed back pressure versus frozen outlet area")
plt.show()


### Discussion task

Why is it dangerous to interpret every colored point in the efficiency map as an equally reliable operating point? Discuss choking, off-design incidence, nozzle over/under-expansion, shocks, and the empirical loss-correlation range.


## 9. Optimal-speed line

For each pressure ratio, the map can be searched for the speed with maximum modeled efficiency. This produces a control-oriented optimum-speed line:

\[
n_{opt}=f(\Pi).
\]

It is a result of the map, not an independent physical constraint. A fixed-speed machine can be compared with this line but cannot automatically follow it.


In [ ]:
valid_points = map_df.loc[map_df["valid"] & map_df["eta_turb"].notna()].copy()

idx_opt = valid_points.groupby("pressure_ratio")["eta_turb"].idxmax()
opt_line = valid_points.loc[idx_opt].sort_values("pressure_ratio").reset_index(drop=True)

poly_degree = min(3, max(1, len(opt_line) - 1))
poly_coeff = np.polyfit(
    opt_line["pressure_ratio"],
    opt_line["n_rpm"],
    deg=poly_degree,
)
opt_line["n_fit_rpm"] = np.polyval(poly_coeff, opt_line["pressure_ratio"])

fig, ax = plt.subplots()
ax.plot(opt_line["pressure_ratio"], opt_line["n_rpm"], marker="o", label="Map optimum")
ax.plot(opt_line["pressure_ratio"], opt_line["n_fit_rpm"], linestyle="--", label=f"Polynomial degree {poly_degree}")
ax.axhline(n_design_rpm, linewidth=1.0, label="Design/fixed speed")
ax.set_xlabel(r"Pressure ratio $\Pi$ [-]")
ax.set_ylabel(r"Optimal rotational speed $n_{opt}$ [rpm]")
ax.set_title("Modeled optimal-speed line")
ax.legend()
plt.show()

print("Polynomial coefficients, highest power first:")
print(poly_coeff)


## 10. Interpolate the reduced map

`LinearNDInterpolator` is used here because it naturally ignores points excluded from the valid operating region.

A TESPy-side wrapper can query the map using normalized pressure ratio and speed and obtain efficiency, relative power, and the predicted swallowing capacity.


In [ ]:
interp_source = map_df.loc[
    map_df["valid"]
    & map_df["eta_turb"].notna()
    & map_df["P_mech_rel"].notna()
    & map_df["m_dot_rel"].notna()
].copy()

points = interp_source[["PR_rel", "n_rel"]].to_numpy()
eta_interpolator = LinearNDInterpolator(points, interp_source["eta_turb"].to_numpy())
power_interpolator = LinearNDInterpolator(points, interp_source["P_mech_rel"].to_numpy())
mass_flow_interpolator = LinearNDInterpolator(points, interp_source["m_dot_rel"].to_numpy())


def query_reduced_map(PR_rel: float, n_rel: float) -> dict[str, float]:
    eta = float(eta_interpolator(PR_rel, n_rel))
    P_rel = float(power_interpolator(PR_rel, n_rel))
    m_rel = float(mass_flow_interpolator(PR_rel, n_rel))
    if not np.isfinite([eta, P_rel, m_rel]).all():
        raise ValueError("Requested point lies outside the valid interpolation region.")
    return {
        "eta_turb": eta,
        "P_mech_rel": P_rel,
        "m_dot_rel": m_rel,
    }


query_reduced_map(PR_rel=1.0, n_rel=1.0)


## 11. Export contract for TESPy

The long-format CSV contains both dimensional and normalized columns. For the next-day cycle workshop, the minimum useful handoff is:

| Input/query coordinate | Returned value or diagnostic |
|---|---|
| `PR_rel` | `eta_turb` |
| `n_rel` | `P_mech_rel` |
|  | `m_dot_rel` |
|  | `choked` |
|  | `valid` |
|  | `area_mismatch_rel` |

In a cycle model, the predicted `m_dot_rel` can be compared with the cycle mass flow. A mismatch may be handled as a feasibility check, a residual, or a controller target depending on the TESPy implementation.


In [ ]:
export_columns = [
    "pressure_ratio",
    "n_rpm",
    "p_in_Pa",
    "T_in_K",
    "superheat_K",
    "p_out_Pa",
    "m_dot_kg_s",
    "eta_turb",
    "P_mech_W",
    "T_out_K",
    "Ma_nozzle_out_is",
    "Ma_nozzle_out",
    "U_over_c_is",
    "c2u_m_s",
    "area_mismatch_rel",
    "PR_rel",
    "n_rel",
    "m_dot_rel",
    "P_mech_rel",
    "eta_rel",
    "choked",
    "valid",
]

map_export = map_df[export_columns].copy()
map_export.to_csv(OUTPUT_DIR / "turbine_offdesign_map.csv", index=False)
opt_line.to_csv(OUTPUT_DIR / "turbine_optimal_speed_line.csv", index=False)

metadata = {
    "model_version": design["model_version"],
    "fluid": fluid,
    "normalization": {
        "pressure_ratio_design": PR_design,
        "n_design_rpm": n_design_rpm,
        "m_dot_design_kg_s": m_dot_design_kg_s,
        "P_mech_design_W": P_design_W,
        "eta_design": eta_design,
    },
    "map_assumptions": {
        "fixed_geometry": True,
        "fixed_inlet_temperature": False,
        "fixed_superheat_above_dew_line": True,
        "fixed_outlet_pressure": True,
        "mass_flow_from_choked_throat_capacity": True,
        "pressure_ratio_varied_by_inlet_pressure": True,
    },
    "optimal_speed_polynomial_coefficients": poly_coeff.tolist(),
}

with (OUTPUT_DIR / "turbine_map_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Created:")
print(" -", OUTPUT_DIR / "turbine_offdesign_map.csv")
print(" -", OUTPUT_DIR / "turbine_optimal_speed_line.csv")
print(" -", OUTPUT_DIR / "turbine_map_metadata.json")
print(" -", OUTPUT_DIR / "offdesign_efficiency_map.png")


## 12. Suggested student extensions

Choose one extension rather than attempting all of them during the three-hour tutorial:

1. compare fixed-speed operation with the optimum-speed line;
2. add a validity mask based on an acceptable outlet-area mismatch;
3. vary inlet temperature and introduce a corrected mass-flow coordinate;
4. compare the map with a constant-efficiency turbine assumption;
5. calculate a power map and torque map;
6. write a minimal TESPy-facing function that returns efficiency from `PR_rel` and `n_rel`;
7. investigate how partial admission changes the map.

## Limitations of the normalized map

A map normalized only by design pressure ratio and speed is convenient but not automatically transferable to a completely different turbine or working fluid. The final lesson should distinguish:

- normalization for interpolation around one machine;
- similarity/corrected quantities for physically broader scaling;
- true recalculation of the turbine for another fluid or geometry.
